# Your own few-shot classification model for Plant Disease Classification

We are going to implement the [Prototypical Networks](https://arxiv.org/abs/1703.05175). It's the favorite method of many few-shot learning researchers (~2000 citations in 3 years), because 
1) it works well, and
2) it's incredibly easy to grasp and to implement.

In [ ]:
import os
import glob
import random
import itertools

from PIL import Image
from tqdm import tqdm
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

from transformers import AutoImageProcessor, AutoModel

from typing import Dict, Iterator, List, Tuple, Union

import torch
from torch import nn, optim, Tensor
from torch.utils.data import DataLoader, Dataset, Subset, Sampler

import torchvision
from torchvision.transforms import v2
from torchvision import transforms

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

## Defining All support functions
### 1. Confusion Matrix

In [ ]:
def confusion_matrix_heatmap(confusion_matrix, classes):
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix, annot=True, cmap='Blues', fmt='g', xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')

### 2. Plotting Images for N way K shot learning task

In [ ]:
# Copied from easyfsl
def plot_images(images: Tensor, title: str, images_per_row: int):
    """
    Plot images in a grid.
    Args:
        images: 4D mini-batch Tensor of shape (B x C x H x W)
        title: title of the figure to plot
        images_per_row: number of images in each row of the grid
    """
    plt.figure()
    plt.title(title)
    plt.imshow(
        torchvision.utils.make_grid(
            images, nrow=images_per_row).permute(1, 2, 0)
    )

### 3. Custom Dataset class

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None, augmentations=None):
        """
        Initialize dataset.

        Args:
            root_dir (str): Root directory containing class subdirectories.
            transform (callable): Transformations to apply to all images.
            augmentations (list of callable): List of augmentation transforms to apply (None for no augmentations).
        """
        self.root_dir = root_dir
        self.transform = transform
        self.augmentations = augmentations if isinstance(augmentations, list) else [augmentations]
        self.classes = sorted(os.listdir(root_dir))
        self.index_to_class = {i: class_name for i, class_name in enumerate(self.classes)}
        self.image_paths = []
        self.labels = []

        # Populate image paths and labels
        for label, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)
            class_image_paths = glob.glob(os.path.join(class_dir, '*.png')) + \
                                glob.glob(os.path.join(class_dir, '*.jpg')) + \
                                glob.glob(os.path.join(class_dir, '*.jpeg'))
            self.image_paths.extend(class_image_paths)
            self.labels.extend([label] * len(class_image_paths))

    def __len__(self):
        if self.augmentations:
            return len(self.image_paths) * len(self.augmentations)
        return len(self.image_paths)

    def __getitem__(self, idx):
        if self.augmentations:
            base_idx = idx // len(self.augmentations)
            augment_idx = idx % len(self.augmentations)
        else:
            base_idx = idx
            augment_idx = None

        img_path = self.image_paths[base_idx]
        label = self.labels[base_idx]

        # Load and transform image
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        # Apply augmentation if specified
        if augment_idx is not None and self.augmentations[augment_idx]:
            image = self.augmentations[augment_idx](image)

        return image, label

    def get_class_label_mapper(self):
        return self.index_to_class

    def get_labels(self):
        return self.labels

### 4. Subset Creating

In [ ]:
class CustomSubset(Subset):
    def __getattr__(self, attr):
        # Delegate attribute access to the original dataset
        return getattr(self.dataset, attr)

### 5. Sampler for N-way K-shot learning

In [ ]:
class TaskSampler(Sampler):
    def __init__(
        self,
        dataset: Union[CustomDataset, Subset],
        n_way: int,
        n_shot: int,
        n_query: int,
        n_tasks: int,
    ):
        super().__init__(data_source=None)
        self.dataset = dataset
        self.n_way = n_way
        self.n_shot = n_shot
        self.n_query = n_query
        self.n_tasks = n_tasks

        # Handle Subset or CustomDataset
        if isinstance(dataset, Subset):
            print("case 1")
            labels = [dataset.dataset.get_labels()[idx] for idx in dataset.indices]
        else:
            print("case 2")
            labels = dataset.get_labels()

        # Group image indices by label
        self.items_per_label: Dict[int, List[int]] = {}
        for idx, label in enumerate(labels):
            if label not in self.items_per_label:
                self.items_per_label[label] = []
            self.items_per_label[label].append(idx)

        self._validate_sampling_requirements()

    def __len__(self) -> int:
        return self.n_tasks

    def __iter__(self) -> Iterator[List[int]]:
        for _ in range(self.n_tasks):
            sampled_classes = random.sample(self.items_per_label.keys(), self.n_way)
            task_indices = []
            for cls in sampled_classes:
                class_indices = self.items_per_label[cls]
                sampled_indices = random.sample(class_indices, self.n_shot + self.n_query)
                task_indices.extend(sampled_indices)
            yield task_indices

    def episodic_collate_fn(
        self, input_data: List[Tuple[Tensor, Union[Tensor, int]]]
    ) -> Tuple[Tensor, Tensor, Tensor, Tensor, List[int]]:
        input_data = [(img, int(label)) for img, label in input_data]
        true_class_ids = list({label for _, label in input_data})
        all_images = torch.stack([img for img, _ in input_data])
        all_labels = torch.tensor(
            [true_class_ids.index(label) for _, label in input_data]
        )

        # Reshape into (n_way, n_shot + n_query)
        all_images = all_images.view(self.n_way, self.n_shot + self.n_query, *all_images.shape[1:])
        all_labels = all_labels.view(self.n_way, self.n_shot + self.n_query)

        support_images = all_images[:, :self.n_shot].reshape(-1, *all_images.shape[2:])
        query_images = all_images[:, self.n_shot:].reshape(-1, *all_images.shape[2:])
        support_labels = all_labels[:, :self.n_shot].flatten()
        query_labels = all_labels[:, self.n_shot:].flatten()

        return support_images, support_labels, query_images, query_labels, true_class_ids

    def _validate_sampling_requirements(self):
        for label, indices in self.items_per_label.items():
            if len(indices) < self.n_shot + self.n_query:
                raise ValueError(
                    f"Class {label} has only {len(indices)} samples, "
                    f"but at least {self.n_shot + self.n_query} are required."
                )


### 6. Earley Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, save_path="/kaggle/working/best_model.pth"):
        """
        Args:
            patience (int): Number of epochs to wait for improvement.
            min_delta (float): Minimum change in the monitored metric to qualify as an improvement.
            save_path (str): Path to save the best model.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.save_path = save_path

    def check(self, validation_loss, model):
        """
        Check if training should stop.

        Args:
            validation_loss (float): Current validation loss.
            model (torch.nn.Module): Current model instance.
        Returns:
            bool: True if training should stop, False otherwise.
        """
        if validation_loss < self.best_loss - self.min_delta:
            self.best_loss = validation_loss
            self.counter = 0
            # Save the best model
            torch.save(model.state_dict(), self.save_path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

# Training with early stopping
def fit_with_early_stopping(
    train_loader, val_loader, model, optimizer, criterion, num_epochs=100, patience=5
):
    early_stopping = EarlyStopping(patience=patience, save_path="/kaggle/working/best_model.pth")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for episode_index, (
            support_images,
            support_labels,
            query_images,
            query_labels,
            class_ids,
        ) in tqdm(enumerate(train_loader), total=len(train_loader)):
            optimizer.zero_grad()
            classification_scores = model(
                support_images.to(device),
                support_labels.to(device),
                query_images.to(device),
            )

            loss = criterion(classification_scores, query_labels.to(device))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Evaluate on the validation dataset
        val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for episode_index, (
                support_images,
                support_labels,
                query_images,
                query_labels,
                class_ids,
            ) in tqdm(enumerate(val_loader), total=len(val_loader)):
                classification_scores = model(
                    support_images.to(device),
                    support_labels.to(device),
                    query_images.to(device),
                )
                loss = criterion(classification_scores, query_labels.to(device))
                val_loss += loss.item()

        # Average losses
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        # Check early stopping
        if early_stopping.check(val_loss, model):
            print("Early stopping triggered. Restoring the best model.")
            model.load_state_dict(torch.load("best_model.pth"))
            break

    return model


### 7. Evaluation functions for few shot learning

In [ ]:
def evaluate_on_one_task(
    support_images: torch.Tensor,
    support_labels: torch.Tensor,
    query_images: torch.Tensor,
    query_labels: torch.Tensor,
) -> (torch.Tensor, torch.Tensor):
    """
    Returns the model's predictions and the true labels for the query images.
    """
    # Get model predictions
    predictions = torch.max(
        model(support_images.to(device), support_labels.to(device), query_images.to(device))
        .detach()
        .data,
        1,
    )[1]
    
    return predictions.cpu(), query_labels.cpu()


def evaluate(data_loader: DataLoader):
    all_predictions = []
    all_labels = []

    # Put the model in evaluation mode
    model.eval()
    with torch.no_grad():
        for episode_index, (
            support_images,
            support_labels,
            query_images,
            query_labels,
            class_ids,
        ) in tqdm(enumerate(data_loader), total=len(data_loader)):
            
            predictions, labels = evaluate_on_one_task(
                support_images, support_labels, query_images, query_labels
            )

            # Append the predictions and true labels
            all_predictions.append(predictions)
            all_labels.append(labels)

    # Concatenate all predictions and labels for final evaluation
    all_predictions = torch.cat(all_predictions)
    all_labels = torch.cat(all_labels)
    
    # Print accuracy
    accuracy = (all_predictions == all_labels).float().mean().item()
    print(f"Model tested on {len(data_loader)} tasks. Accuracy: {accuracy * 100:.2f}%")
    
    # Generate classification report and confusion matrix
    print("\nClassification Report:\n", classification_report(all_labels, all_predictions))
    confusion = confusion_matrix(all_labels, all_predictions)
    confusion_matrix_heatmap(confusion, classes)

In [ ]:
# copied from easyfsl
def sliding_average(value_list: List[float], window: int) -> float:
    """
    Computes the average of the latest instances in a list
    Args:
        value_list: input list of floats (can't be empty)
        window: number of instances to take into account. If value is 0 or greater than
            the length of value_list, all instances will be taken into account.

    Returns:
        average of the last window instances in value_list

    Raises:
        ValueError: if the input list is empty
    """
    if len(value_list) == 0:
        raise ValueError("Cannot perform sliding average on an empty list.")
    return np.asarray(value_list[-window:]).mean()

In [ ]:
image_size = 224
basic_transformations = [
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
]

final_transformation = [
    v2.RandomResizedCrop(image_size, scale=(0.75, 1.0), ratio=(1.0, 1.0))
]

### 8. Splitting Train, Vlidation adn Test datasets

In [ ]:
def _accumulate(iterable, fn=lambda x, y: x + y):
    "Return running totals"
    # _accumulate([1,2,3,4,5]) --> 1 3 6 10 15
    # _accumulate([1,2,3,4,5], operator.mul) --> 1 2 6 24 120
    it = iter(iterable)
    try:
        total = next(it)
    except StopIteration:
        return
    yield total
    for element in it:
        total = fn(total, element)
        yield total

In [ ]:
def custom_random_split(dataset, lengths, generator=None):
    indices = torch.randperm(sum(lengths), generator=generator).tolist()
    return [CustomSubset(dataset, indices[offset - length:offset]) for offset, length in zip(_accumulate(lengths), lengths)]

In [ ]:
def make_ds(dataset_dir, filtered_dataset_dir, class_map, dataset_name, target_size=(224, 224)):
    """
    Resizes and organizes images from the source dataset directory into a filtered dataset directory.

    Args:
        dataset_dir (str): Source directory containing images organized in class subfolders.
        filtered_dataset_dir (str): Target directory where resized and renamed images will be stored.
        class_map (dict): Mapping of source class names to target class names.
        dataset_name (str): Prefix for the renamed image files.
        target_size (tuple): Target size for resizing images (width, height).
    """
    if not os.path.exists(filtered_dataset_dir):
        os.makedirs(filtered_dataset_dir)

    for class_name, target_class_name in class_map.items():
        src = os.path.join(dataset_dir, class_name)
        dst = os.path.join(filtered_dataset_dir, target_class_name)

        if os.path.exists(src):
            if not os.path.exists(dst):
                os.makedirs(dst)

            image_files = [
                f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))
            ]
            
            # Determine the last used index in the destination folder
            existing_files = glob.glob(os.path.join(dst, f"{dataset_name}_{class_name}_*."))
            if existing_files:
                last_index = max(
                    int(os.path.basename(f).split("_")[-1].split(".")[0])
                    for f in existing_files
                )
            else:
                last_index = -1

            for i, image in enumerate(image_files, start=last_index + 1):
                try:
                    ext = image.split(".")[-1].lower()
                    if ext not in ["jpg", "jpeg", "png"]:
                        print(f"Skipping unsupported file format: {image}")
                        continue

                    # Define the new image path in the destination folder
                    new_name = f"{dataset_name}_{class_name}_{i}.{ext}"
                    new_path = os.path.join(dst, new_name)
                    
                    # Open, resize, and save the image to the destination
                    img_path = os.path.join(src, image)
                    with Image.open(img_path) as img:
                        img = img.resize(target_size, Image.Resampling.LANCZOS)
                        img.save(new_path)
                    
                except Exception as e:
                    print(f"Error processing {image}: {e}")
        else:
            print(f"Source directory {src} does not exist. Skipping...")

    print(f"Data has been resized and copied to {filtered_dataset_dir}")

In [ ]:
plantvillage_dir = "/kaggle/input/plantvillage-dataset/color"
plantdoc_train_dir = "/kaggle/input/plantdoc-dataset/train"
plantdoc_test_dir = "/kaggle/input/plantdoc-dataset/test"
filtered_train_dir = '/kaggle/working/data/train'
filtered_test_dir = '/kaggle/working/data/test'


classes = ['Apple___healthy', 'Apple___rust',
            'Bell_pepper___bacterial_spot', 'Bell_pepper___healthy',
           'Grape___black_rot', 'Grape___healthy',
          'Tomato___healthy', 'Tomato___septoria_leaf_spot' ]

plant_village_map = {
    'Apple___healthy': 'Apple___healthy',
    'Apple___Cedar_apple_rust': 'Apple___rust',
    'Pepper,_bell___Bacterial_spot': 'Bell_pepper___bacterial_spot',
    'Pepper,_bell___healthy': 'Bell_pepper___healthy',
    'Grape___Black_rot': 'Grape___black_rot',
    'Grape___healthy': 'Grape___healthy',
    'Tomato___healthy': 'Tomato___healthy',
    'Tomato___Septoria_leaf_spot': 'Tomato___septoria_leaf_spot'
}

plantdoc_map = {
    'Apple_leaf': 'Apple___healthy', 
    'Apple_rust_leaf' : 'Apple___rust',
    'Bell_pepper_leaf': 'Bell_pepper___healthy',
    'Bell_pepper_leaf_spot': 'Bell_pepper___bacterial_spot',
    'grape_leaf': 'Grape___healthy',
    'grape_leaf_black_rot': 'Grape___black_rot',
    'Tomato_leaf': 'Tomato___healthy',
    'Tomato_Septoria_leaf_spot': 'Tomato___septoria_leaf_spot'
}


label_mapping = {i: class_name for i, class_name in enumerate(sorted(classes))}

make_ds(plantvillage_dir, filtered_train_dir, plant_village_map, 'plantvillage')
make_ds(plantdoc_train_dir, filtered_test_dir, plantdoc_map, 'plantdoc')
make_ds(plantdoc_test_dir, filtered_test_dir, plantdoc_map, 'plantdoc')

In [ ]:
# Create base dataset
base_dataset = CustomDataset(filtered_train_dir, transform=v2.Compose(basic_transformations))

# Train, val, and test splits
train_size = int(0.7 * len(base_dataset))
test_size = int(0.1 * len(base_dataset))
val_size = len(base_dataset) - (train_size + test_size)

train_dataset, val_dataset, test_dataset = custom_random_split(base_dataset, [train_size, val_size, test_size])

# Apply augmentations
train_augmentations = v2.Compose(final_transformation)
train_dataset.dataset.augmentations = [train_augmentations]
val_dataset.dataset.augmentations = [v2.Compose(final_transformation)]
test_dataset.dataset.augmentations = [v2.Compose(final_transformation)]

# Target dataset
target_dataset = CustomDataset(
    filtered_test_dir,
    transform=v2.Compose(basic_transformations + final_transformation)
)

In a few-shot classification task, you have a labeled support set (which kind of acts like a catalog) and query set. For each image of the query set, we want to predict a label from the labels present in the support set. A few-shot classification model has to use the information from the
support set in order to classify query images. We say *few-shot* when the support set contains very few images for each label (typically less than 10). 

Most few-shot classification methods are *metric-based*. It works in two phases : 
1) they use a CNN to project both support and query images into a feature space, and
2) they classify query images by comparing them to support images.
If, in the feature space, an image is closer to one class than it is to other classes, we will guess that it's a that class.

From there, we have two challenges :

1. Find the good feature space. This is what convolutional networks are for. A CNN is basically a function that takes an image as input and outputs a representation (or *embedding*) of this image in a given feature space. The challenge here is to have a CNN that will project images of the same class into representations that are close to each other, even if it has not been trained
on objects of this class.
2. Find a good way to compare the representations in the feature space. This is the job of Prototypical Networks.

From the support set, Prototypical Networks compute a prototype for each class, which is *the mean of all embeddings of support images from this class*. Then, each query is simply classified as the nearest prototype in the feature space, with respect to euclidean distance.

If you want to learn more about how this works, it has been exlpained in [here](https://www.sicara.fr/blog-technique/few-shot-image-classification-meta-learning). But now, let's get to coding. In the code below (modified from [this](https://github.com/sicara/easy-few-shot-learning/blob/master/easyfsl/methods/prototypical_networks.py)), we simply define Prototypical Networks as a torch module, with a `forward()` method. You may notice 2 things.

1. We initiate `PrototypicalNetworks` with a *backbone*. This is the feature extractor we were talking about. Here, we use as backbone a EfficientNet B0 pretrained on `PlantVillage dataset`, with its head chopped off and replaced by a `Flatten` layer. The output of the backbone, for an input image, will be a 1280-dimensional feature vector.
2. The forward method doesn't only take one input tensor, but 3: in order to predict the labels of query images, we also need support images and labels as inputs of the model.

## Defining Models

In [ ]:
NUM_CLASSES = len(classes)

In [ ]:
class PrototypicalNetworks(nn.Module):
    def __init__(self, backbone: nn.Module):
        super(PrototypicalNetworks, self).__init__()
        self.backbone = backbone

    def forward(
        self,
        support_images: torch.Tensor,
        support_labels: torch.Tensor,
        query_images: torch.Tensor,
    ) -> torch.Tensor:
        """
        Predict query labels using labeled support images.
        """
        # Extract features for support and query images
        z_support = self.backbone(support_images).last_hidden_state  
        z_query = self.backbone(query_images).last_hidden_state

        # Ensure the feature tensors are 2D (batch_size, feature_dim)
        if z_support.dim() > 2:
            z_support = z_support.view(z_support.size(0), -1)
        if z_query.dim() > 2:
            z_query = z_query.view(z_query.size(0), -1)

        # Infer the number of classes from the labels of the support set
        n_way = len(torch.unique(support_labels))

        # Compute class prototypes (mean of support features per class)
        z_proto = torch.stack([
            z_support[support_labels == label].mean(dim=0)
            for label in range(n_way)
        ])

        # Compute the euclidean distance between queries and prototypes
        dists = torch.cdist(z_query, z_proto)

        # Convert distances to classification scores (negative distance for similarity)
        scores = -dists
        return scores


# Instantiate the model
processor = AutoImageProcessor.from_pretrained('/kaggle/input/dinov2/pytorch/small/1/')
backbone = AutoModel.from_pretrained('/kaggle/input/dinov2/pytorch/small/1/')

# Wrap the backbone in PrototypicalNetworks
model = PrototypicalNetworks(backbone).to(device)

Now we have a model! Note that we used a pretrained feature extractor, so our model should already be up and running. Let's see that.

Here we create a dataloader that will feed few-shot classification tasks to our model.
But a regular PyTorch dataloader will feed batches of images, with no consideration for their label or whether they are support or query. We need 2 specific features in our case.

1. We need images evenly distributed between a given number of classes.
2. We need them split between support and query sets.

For the first point, I wrote a custom sampler: it first samples `n_way` classes from the dataset, then it samples `n_shot + n_query` images for each class (for a total of `n_way * (n_shot + n_query)` images in each batch).
For the second point, I have a custom collate function to replace the built-in PyTorch `collate_fn`.
This baby feed each batch as the combination of 5 items:

1. support images
2. support labels between 0 and `n_way`
3. query images
4. query labels between 0 and `n_way`
5. a mapping of each label in `range(n_way)` to its true class id in the dataset
(it's not used by the model but it's very useful for us to know what the true class is)

You can see that in PyTorch, a DataLoader is basically the combination of a sampler, a dataset and a collate function (and some multiprocessing voodoo): sampler says which items to fetch, the dataset says how to fetch them, and the collate function says how to present these items together. If you want to dive into these custom objects, they're [here](https://github.com/sicara/easy-few-shot-learning/tree/master/easyfsl/data_tools).

## 8 way 10 shot learning for 100 Evaluation tasks for test portion of plant village dataset

In [ ]:
N_WAY = NUM_CLASSES  # Number of classes per task
N_SHOT = 10  # Support images per class
N_QUERY = 10  # Query images per class
N_EVALUATION_TASKS = 100

test_sampler = TaskSampler(
    test_dataset, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY, n_tasks=N_EVALUATION_TASKS
)

test_loader = DataLoader(
    test_dataset,
    batch_sampler=test_sampler,
    pin_memory=True,
    collate_fn=test_sampler.episodic_collate_fn,
)

(
    example_support_images,
    example_support_labels,
    example_query_images,
    example_query_labels,
    example_class_ids,
) = next(iter(test_loader))

plot_images(example_support_images, "support images", images_per_row=N_SHOT)
plot_images(example_query_images, "query images", images_per_row=N_QUERY)

In [ ]:
# Run the evaluation
evaluate(test_loader)

## 8 way 10 shot learning for 100 Evaluation tasts for plantDoc dataset

In [ ]:
N_WAY = NUM_CLASSES  # Number of classes per task
N_SHOT = 10  # Support images per class
N_QUERY = 10  # Query images per class
N_EVALUATION_TASKS = 100

target_sampler = TaskSampler(
    target_dataset, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY, n_tasks=N_EVALUATION_TASKS
)

target_loader = DataLoader(
    target_dataset,
    batch_sampler=target_sampler,
    pin_memory=True,
    collate_fn=test_sampler.episodic_collate_fn,
)

(
    example_support_images,
    example_support_labels,
    example_query_images,
    example_query_labels,
    example_class_ids,
) = next(iter(target_loader))

plot_images(example_support_images, "support images", images_per_row=N_SHOT)
plot_images(example_query_images, "query images", images_per_row=N_QUERY)

In [ ]:
evaluate(target_loader)

## Training a meta-learning algorithm

In 1998, Thrun & Pratt stated that, given one task to solve, an algorithm is learning “if its performance at the task improves with experience”, while, given a family of tasks to solve, an algorithm is learning to learn if “its performance at each task improves with experience and with the number of tasks”. We will refer to the last one as a meta-learning algorithm. It doesn’t learn how to solve a specific task. It successively learns to solve many tasks. And each time it learns a new task, it becomes better at learning new tasks: it learns to learn.

Let's use the PlantVillage dataset as training set. Here we prepare a data loader of 10 000 few-shot classification tasks on which we will train our model. We use PlantDoc dataset for test our trained model
This guarantees that at test time, the model will have to classify characters that were not seen during training.

Note that we don't set a validation set here to keep this notebook concise, but keep in mind that **this is not good practice** and you should always use validation when training a model for production.

## Training a prototypical network using 8 way 5 shot learning using plant village dataset

In [ ]:
N_WAY = NUM_CLASSES  # Number of classes per task
N_SHOT = 10  # Support images per class
N_QUERY = 10  # Query images per class

N_TRAINING_EPISODES = 1000
N_VALIDATION_TASKS = 100

train_sampler = TaskSampler(
    train_dataset, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY, n_tasks=N_TRAINING_EPISODES
)
train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    pin_memory=True,
    collate_fn=train_sampler.episodic_collate_fn,
)

val_sampler = TaskSampler(
    val_dataset, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY, n_tasks=N_TRAINING_EPISODES
)
val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_sampler,
    pin_memory=True,
    collate_fn=val_sampler.episodic_collate_fn,
)

We will keep the same model. So our weights will be pre-trained on 8 classes of PlantVillage dataset. If you want to start a training from scratch,
feel free to set `pretrained=False` in the definition of the EfficientNet.

Here we define our loss and our optimizer (cross entropy and Adam, pretty standard), and a `fit` method. 
This method takes a classification task as input (support set and query set). It predicts the labels of the query set based on the information from the support set; then it compares the predicted labels to ground truth query labels, and this gives us a loss value. Then it uses this loss to update the parameters of the model. This is a *meta-training loop*.

In [ ]:
# Assuming train_loader and val_loader are DataLoader objects
model = PrototypicalNetworks(backbone).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = fit_with_early_stopping(
    train_loader=train_loader,
    val_loader=val_loader,
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=50,
    patience=3,
)


To train the model, we are just going to iterate over a large number of randomly generated few-shot classification tasks, and let the `fit` method update our model after each task. This is called **episodic training**.

In [ ]:
evaluate(test_loader)

In [ ]:
N_EVALUATION_TASKS = 100

target_sampler = TaskSampler(
    target_dataset, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY, n_tasks=N_EVALUATION_TASKS
)

target_loader = DataLoader(
    target_dataset,
    batch_sampler=target_sampler,
    pin_memory=True,
    collate_fn=target_sampler.episodic_collate_fn,
)

evaluate(target_loader)